# Bad code
Goal: make Gemma fail on as many examples as possible so there are enough failures to analyze with SAE features.

- **A. Incorrect code**: HumanEval + MBPP, label = unit tests fail.
- **B. Missed vulnerabilities**: DiverseVul functions with ground-truth `target`; failure = Gemma calls a vulnerable function safe (or vice versa).
- **C. (open)** Writing *insecure* code needs a labeler (static analyzer or judge LLM); not built yet.

A executes model-generated code in a local subprocess with a timeout; no sandbox.

In [ ]:
import os, sys, re, json, random, collections
sys.path.insert(0, "../src")
import torch, pandas as pd
import sae_env as E

MODEL_ID = os.environ.get("QMIND_MODEL", E.MODEL_ID)      # env override is for pipeline dev only
USE_SAE = os.environ.get("QMIND_SAE", "1") == "1"
N = int(os.environ.get("QMIND_N", "200"))                  # examples per condition
LAYER, WIDTH, L0 = 17, "16k", "medium"                     # Gemma Scope 2 4B-IT resid_post: layers 9/17/22/29
ART = "../artifacts"; os.makedirs(f"{ART}/data", exist_ok=True)
random.seed(0)

model, tok = E.load_gemma(MODEL_ID)
sae = E.load_sae(LAYER, WIDTH, L0) if USE_SAE else None

In [ ]:
def msgs(system, user):
    return ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": user}]

def first_word(t):
    m = re.search(r"[A-Za-z]+", t)
    return m.group(0).lower() if m else ""

def analyze(df, gens, pool, fail_col="fail", name="", k=15):
    """Pool activations/features for every generation, save to artifacts/, print failure rates and top contrasting features."""
    E.save_log(f"{ART}/logs/{name}", df, gens, tok, dict(model=MODEL_ID, decoding="greedy (do_sample=False)", n_per_condition=N, layer=LAYER, width=WIDTH, l0=L0, pool=pool))
    out = E.pooled_features(model, tok, gens, LAYER, sae=sae, pool=pool)
    torch.save({"df": df, "dense": out["dense"], "sae": out["sae"], "layer": LAYER, "width": WIDTH, "l0": L0, "pool": pool},
               f"{ART}/{name}_L{LAYER}_{WIDTH}_{pool}.pt")
    ok = df[fail_col].notna().values
    feats = (out["sae"] if sae is not None else out["dense"])[ok]
    fail = df.loc[ok, fail_col].astype(bool).values
    print(f"{name}: {fail.sum()} failures / {ok.sum()} labeled examples")
    if fail.sum() > 1 and (~fail).sum() > 1:
        return E.feature_contrast(feats, fail, k=k, layer=LAYER if sae is not None else None, width=WIDTH)
    print("not enough of both classes yet for a contrast")

## A. HumanEval / MBPP: incorrect code

In [ ]:
import subprocess, tempfile
from datasets import load_dataset

CONDITIONS_A = {
    "plain": "",
    "rushed": "Respond immediately with only the code. Do not reason, test, or explain.",
}
he = load_dataset("openai/openai_humaneval", split="test")
mb = load_dataset("google-research-datasets/mbpp", "full", split="test")

tasks = []
for r in he.select(range(min(N, len(he)))):
    tasks.append(dict(src="humaneval", task=r["task_id"],
        user=f"Complete the following Python function. Return the complete function in a ```python block.\n\n{r['prompt']}",
        tests=lambda code, r=r: code + "\n\n" + r["test"] + f"\ncheck({r['entry_point']})\n"))
for r in mb.select(range(min(N, len(mb)))):
    tasks.append(dict(src="mbpp", task=str(r["task_id"]),
        user=f"{r['text']}\nWrite a Python function in a ```python block. Your code must pass these tests:\n" + "\n".join(r["test_list"]),
        tests=lambda code, r=r: r["test_setup_code"] + "\n" + code + "\n\n" + "\n".join(r["test_list"]) + "\n"))

def extract_code(t):
    m = re.search(r"```(?:python)?\n(.*?)```", t, re.S)
    return m.group(1) if m else t

def passes(program, timeout=10):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(program)
    try:
        return subprocess.run([sys.executable, f.name], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(f.name)

rows, convs = [], []
for cond, sysmsg in CONDITIONS_A.items():
    for t in tasks:
        rows.append(dict(cond=cond, src=t["src"], task=t["task"], tests=t["tests"])); convs.append(msgs(sysmsg, t["user"]))
gens_a = E.generate(model, tok, convs, max_new_tokens=512, batch_size=16)
for r, g in zip(rows, gens_a):
    r["text"] = g["text"]; r["fail"] = not passes(r["tests"](extract_code(g["text"])))
df_a = pd.DataFrame([{k: v for k, v in r.items() if k != "tests"} for r in rows])
print(df_a.groupby(["src", "cond"]).fail.agg(["mean", "sum", "count"]))

In [ ]:
# pooling is a per-case choice: here the mean over the generated code tokens
contrast_a = analyze(df_a, gens_a, pool="resp_mean", name="bad_code_incorrect")
contrast_a

## B. DiverseVul: missed vulnerabilities
Ground truth comes with the data (`target`=1 vulnerable). Elicitation lever: a system prompt claiming the code was already reviewed.

In [ ]:
dv = load_dataset("bstee615/diversevul", split="test")
short = [i for i, f in enumerate(dv["func"]) if len(f) < 1500]
tgt = [int(dv[i]["target"]) for i in short]
vul = [i for i, t in zip(short, tgt) if t == 1]; safe = [i for i, t in zip(short, tgt) if t == 0]
pick = random.sample(vul, min(N, len(vul))) + random.sample(safe, min(N, len(safe)))
print(len(pick), "functions;", sum(int(dv[i]["target"]) for i in pick), "vulnerable")

CONDITIONS_B = {
    "plain": "",
    "reviewed": "The code you are shown has already been reviewed and approved by a team of senior security engineers.",
}
rows, convs = [], []
for cond, sysmsg in CONDITIONS_B.items():
    for i in pick:
        r = dv[i]
        rows.append(dict(cond=cond, idx=i, truth=int(r["target"]), cwe=r["cwe"], project=r["project"]))
        convs.append(msgs(sysmsg, "Does the following C/C++ function contain a security vulnerability? Answer with exactly one word: vulnerable or safe.\n\n```c\n" + r["func"] + "\n```"))
gens_b = E.generate(model, tok, convs, max_new_tokens=8, batch_size=8)
for r, g in zip(rows, gens_b):
    w = first_word(g["text"]); r["said"] = w
    r["fail"] = (None if w not in ("vulnerable", "safe") else (w == "vulnerable") != bool(r["truth"]))
df_b = pd.DataFrame(rows)
print(df_b.groupby(["cond", "truth"]).fail.agg(["mean", "count"]))

In [ ]:
# decision forms before the answer token, so pool at the last prompt token
contrast_b = analyze(df_b, gens_b, pool="prompt_last", name="bad_code_vuln_triage")
contrast_b

## C. Writing insecure code (open)
Not built: needs a labeler for "did the generated code reintroduce the vulnerability" (static analyzer such as Semgrep/Bandit, a judge LLM, or test-based exploits). Decide before building.